# 🎵 MoodBeats AI — Session 3: APIs & Google Gemini

Today we teach an AI to detect moods from text using the **Google Gemini API**.

---
## What is an API?

**API** = Application Programming Interface.

Think of a **restaurant**:
- **You** (your code) = the customer
- **Waiter** (API) = the interface — takes your order, brings back food
- **Kitchen** (Google's servers) = processes the request

You never enter the kitchen. You just send a request and get a response.

---
## Step 1: Install the Gemini Library

In [ ]:
!pip install google-generativeai -q
print("Installed!")

## Step 2: Get Your Gemini API Key

1. Go to **https://aistudio.google.com**
2. Sign in with your Google account
3. Click **"Get API Key"** → **"Create API Key"**
4. Copy the key

### Store your key SECURELY in Colab Secrets
1. Click the **🔑 key icon** in the left sidebar
2. Click **"Add new secret"**
3. Name: `GEMINI_API_KEY`  |  Value: paste your key
4. Toggle **"Notebook access"** ON

⚠️ **Never paste your API key directly in a code cell!** Anyone with the notebook can steal it.

In [ ]:
# Load the API key from Colab Secrets
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)
print("API key loaded!")

---
## Step 3: Your First Gemini Call

Let's ask Gemini something simple first.

In [ ]:
MODEL = "gemini-2.5-flash"

response = client.models.generate_content(model=MODEL, contents="Tell me one fun fact about music in one sentence.")
print(response.text)

That worked! Now let's make it useful for MoodBeats.

---
## Part 1: JSON — The Data Format APIs Use

APIs return data as **JSON** (JavaScript Object Notation) — it looks just like Python dicts.

```json
{"emotion": "happy", "confidence": 0.92}
```

Python's `json` module converts between JSON strings and Python dicts.

In [ ]:
import json

# JSON string → Python dict
json_string = '{"emotion": "happy", "confidence": 0.92}'
data = json.loads(json_string)
print(type(data))         # → <class 'dict'>
print(data["emotion"])    # → happy
print(data["confidence"]) # → 0.92

# Python dict → JSON string
back_to_json = json.dumps(data)
print(back_to_json)

---
## Part 2: Building the Mood Analysis Prompt

We ask Gemini to return a **structured JSON response**. The key is telling it exactly what format we want.

A prompt is just a string — but a well-crafted prompt gives predictable results.

In [ ]:
MOOD_PROMPT = """
You are a music therapist AI. Analyse the user's mood from the text below.

Respond with ONLY a valid JSON object — no markdown, no extra text.
Required format:
{{
  "emotion": "<primary emotion word>",
  "confidence": <float 0.0–1.0>,
  "mood_description": "<2-sentence friendly description>",
  "songs": [
    {{"title": "<song>", "artist": "<artist>", "genre": "<genre>", "why": "<1-line reason>"}},
    ... (8 songs total)
  ]
}}

User text: \"{mood_text}\"
"""

def analyze_mood(mood_text):
    """Send mood text to Gemini and return a parsed result dict."""
    prompt = MOOD_PROMPT.format(mood_text=mood_text)
    response = client.models.generate_content(model=MODEL, contents=prompt)
    raw = response.text

    # Strip markdown code fences if present (Gemini sometimes adds ```json ... ```)
    if raw.startswith("```"):
        raw = raw.split("```")[1]         # get content between fences
        if raw.startswith("json"):
            raw = raw[4:]                 # remove the word 'json'

    return json.loads(raw.strip())

# Test it!
result = analyze_mood("I just got great news and can't stop smiling!")
print(f"Emotion:     {result['emotion']}")
print(f"Confidence:  {result['confidence'] * 100:.0f}%")
print(f"Description: {result['mood_description']}")
print(f"\nFirst song:  {result['songs'][0]['title']} — {result['songs'][0]['artist']}")

### ✏️ In-Class Exercise 3a — Test Different Moods

In [ ]:
# TODO: Test analyze_mood() with 3 different mood descriptions
# Pick moods you're curious about!

test_moods = [
    "I haven't slept in days and everything feels overwhelming.",
    # TODO: Add 2 more mood descriptions
]

for mood_text in test_moods:
    print(f"\nInput: {mood_text}")
    result = analyze_mood(mood_text)
    print(f"Emotion: {result['emotion']} ({result['confidence']*100:.0f}% confidence)")
    print("Songs:")
    for song in result["songs"][:3]:  # show first 3
        print(f"  🎵 {song['title']} — {song['artist']}")

---
## Part 3: Error Handling with try/except

APIs can fail: network error, invalid JSON, rate limit hit. Always wrap API calls in `try/except`.

In [ ]:
def safe_analyze_mood(mood_text):
    """Same as analyze_mood but returns a safe default on failure."""
    try:
        return analyze_mood(mood_text)
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        # Return a safe default so the app doesn't crash
        return {
            "emotion": "neutral",
            "confidence": 0.5,
            "mood_description": "Could not analyse mood. Here are some general picks.",
            "songs": []
        }
    except Exception as e:
        print(f"Unexpected error: {e}")
        return {"emotion": "neutral", "confidence": 0.5,
                "mood_description": "Service temporarily unavailable.", "songs": []}

# Test with normal input
result = safe_analyze_mood("Feeling great today!")
print(f"Emotion: {result['emotion']}")

### ✏️ In-Class Exercise 3b — Full Mood Analyser

Put it all together: take user input, call Gemini, display the result neatly.

In [ ]:
def display_result(result):
    """Print the analysis result in a nice format."""
    print("\n" + "="*50)
    print(f"  Emotion:    {result['emotion'].upper()}")
    print(f"  Confidence: {result['confidence']*100:.0f}%")
    print(f"  {result['mood_description']}")
    print("\n  Recommended Songs:")
    for i, song in enumerate(result["songs"], 1):
        print(f"    {i}. {song['title']} — {song['artist']} ({song['genre']})")
    print("="*50 + "\n")

# TODO: Ask the user for their mood description, call safe_analyze_mood, then display_result
user_input = input("Describe how you're feeling: ")
result = safe_analyze_mood(user_input)
display_result(result)

---
## 🏠 After-Class Exercises

---
### After-Class A: Change the Song Count

In [ ]:
# TODO: Modify MOOD_PROMPT to request 5 songs instead of 8
# Then test it and verify exactly 5 songs come back
# Hint: just change the number in the prompt text

### After-Class B: Add a Playlist Name

In [ ]:
# TODO: Modify the prompt to ask Gemini to also return a "playlist_name" key
# Example: {"playlist_name": "Sunshine Vibes", "emotion": "happy", ...}
# Then display the playlist name in display_result()

### After-Class C (Challenge): Robust JSON Extraction

Sometimes Gemini wraps output in markdown. Write a more robust extractor.

In [ ]:
import re

def extract_json(raw_text):
    """Extract the first valid JSON object from raw_text, handling markdown fences."""
    # TODO: Use regex to find text between { and } and try to parse it
    # Hint: re.search(r'\{.*\}', raw_text, re.DOTALL)
    match = re.search(r'\{.*\}', raw_text, re.DOTALL)
    if match:
        return json.loads(match.group())
    raise ValueError("No JSON found in response")

# Test it
test_input = 'Sure! Here is your JSON:\n```json\n{"emotion": "happy"}\n```'
print(extract_json(test_input))

---
## ✅ Session 3 Complete!

**You learned:**
- What an API is and how HTTP request/response works
- How to configure and call Google Gemini
- How to parse JSON responses into Python dicts
- Error handling with try/except

**Next session:** FastAPI — we build the web server that will serve all these results to a browser!